**Bakehouse Customers**

Dataset: samples.bakehouse.sales_customers

Difficulty: Easy

Topics: groupBy, filter, distinct, null handling

In [0]:
from pyspark.sql import functions as F, types as T

**Learn — groupBy, filter, distinct**

**Function	What it does**
- df.groupBy(col).count()	Counts rows per group — the most common aggregation
- df.groupBy(col).agg(F.count("*"))	Equivalent to .count() but lets you add more aggregations
- df.filter(F.col(c).isNull())	Keeps only rows where the column is null
- df.filter(F.col(c).isNotNull())	Keeps only rows where the column has a value
- df.distinct()	Removes duplicate rows
- df.select(col).distinct().count()	Counts unique values in a column

In [0]:

# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_customers")

# Count rows per city (not per country — the problems ask about country)
df.groupBy("city").agg(
    F.count("*").alias("num_customers")
).orderBy(F.col("num_customers").desc()).show(5)

# How many unique email domains exist?
print("Unique emails:", df.select("email_address").distinct().count())

# Check for nulls in a specific column
null_count = df.filter(F.col("city").isNull()).count()
print("Rows with null city:", null_count)

**Problem 1**

Count the number of customers in each country, sorted from most to fewest. Load samples.bakehouse.sales_customers and group by country.

Expected output columns:

country - country name
customer_count - number of customers in that country (sorted descending)

In [0]:
df.printSchema()

In [0]:

# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = df.groupby("country").agg(F.count("customerID").alias("customer_count")).orderBy(F.col("customer_count").desc())
result_1.show()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'country' in cols, "Missing column: country"
assert 'customer_count' in cols, "Missing column: customer_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['customer_count'] for r in result_1.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by customer_count descending"
assert all(c > 0 for c in counts), "All customer counts must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

**Problem 2**

Find all distinct continents represented in the customer dataset. Each continent should appear only once in the result.

Expected output columns:

continent - unique continent name

In [0]:

# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = df.select("continent").distinct()
result_2.show()

In [0]:

# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'continent' in cols, "Missing column: continent"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Too many continents ({cnt}), expected <= 10 distinct values"
continent_vals = [r['continent'] for r in result_2.collect()]
assert len(continent_vals) == len(set(continent_vals)), "continent values must be distinct"
print(f"Problem 2 passed ✓  ({cnt} rows)")